<a href="https://colab.research.google.com/github/marcos-mansur/load-forecast/blob/main/Data_quality.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Objective

Verify data quality
- identify missing days
- input with day before

# load libs

In [ ]:
!pip install pendulum

In [39]:
import tensorflow as tf
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.utils import timeseries_dataset_from_array
import numpy as np
import pandas as pd
import pendulum
#import optuna

#load data 

In [128]:
df_20XX = pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_2001.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])

for x in range(2002,2022):
  df_20XX = pd.concat(objs = (df_20XX,pd.read_csv(f'https://ons-dl-prod-opendata.s3.amazonaws.com/dataset/carga_energia_di/CARGA_ENERGIA_{x}.csv', 
                      sep=';', 
                      parse_dates=['din_instante'])))

load_col = 'val_cargaenergiamwmed'
time_col = 'din_instante'

# treat data

In [126]:
#df.iloc[missing_days.index[1]:]
missing_days.index[1]

5569

In [131]:
go_to_friday(df.iloc[missing_days.index[1]-1:])

,din_instante,val_cargaenergiamwmed,semana,Mes,dia semana,dia mes,ano
0,2016-04-15,41318.77,795,4,Friday,15,2016
1,2016-04-16,37260.27,795,4,Saturday,16,2016
2,2016-04-17,33414.91,795,4,Sunday,17,2016
3,2016-04-18,40487.70,796,4,Monday,18,2016
4,2016-04-19,41375.98,796,4,Tuesday,19,2016
...,...,...,...,...,...,...,...
2082,2021-12-27,39736.50,1093,12,Monday,27,2021
2083,2021-12-28,40201.57,1093,12,Tuesday,28,2021
2084,2021-12-29,40083.16,1093,12,Wednesday,29,2021
2085,2021-12-30,38850.81,1093,12,Thursday,30,2021


In [143]:
def get_missing_days(df):
  # range of every day from 2001 to 2021
  time_delta = pd.date_range(start = df.din_instante.iloc[0], end= df.din_instante.iloc[-1],freq='D')
  # turn into df
  df_time = pd.DataFrame(data={'data':time_delta})
  # left join range of data with datas from dataset, missing days will become NaN
  df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
  # missing days indexes
  df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()]
  # series of missing days with indexes
  missing_days = df_missing.loc[df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()].index].data
  return missing_days

def id_and_impute(df):

  df = df.copy()
  # series of missing days with indexes
  missing_days = get_missing_days(df) 
  # drop days from incomplete week in 2016
  df = df.drop(axis=0, index = df.din_instante[(df[time_col]>='2016-04-01') & (df[time_col]<='2016-04-05')].index)
  df = df.drop(axis=0, index = missing_days.index[1]-1)
  # missing day to be inputed - feb 1st, 2014
  input_day = df['din_instante'].iloc[missing_days.index[0]]
  # line to be inserted in dataset with load value of day before
  df_day = pd.DataFrame({'din_instante': input_day-pd.Timedelta(1, unit='D'),	
                        'val_cargaenergiamwmed': df[load_col].iloc[missing_days.index[0] - 1]},
                        index=[missing_days.index[0] -1 ])
  # insert missing day
  df_total = pd.concat(objs= [df[:missing_days.index[0]], 
                              df_day, 
                              df[missing_days.index[0]:]])
  
  #reset index
  return  df_total, missing_days

def go_to_friday(df): 
  """ get next friday = start the operative week"""
  
  # first day in dataset
  date_time = df['din_instante'].iloc[0]
  # check if the dataset starts on a friday 
  if date_time.day_name() != 'Friday':
    # today
    dt = pendulum.datetime(date_time.year,date_time.month, date_time.day)
    # next friday - begins the operative week
    next_friday = dt.next(pendulum.FRIDAY).strftime('%Y-%m-%d')
    # df starts with the begin of operative week
    return df[df['din_instante'] >= next_friday].reset_index(drop=True).copy()


def treat_data(df,regiao='SUDESTE',operative_week_start=2):
  
  # round the values of load
  df['val_cargaenergiamwmed'] = np.round(df['val_cargaenergiamwmed'],2)
  # drop last 4 rows that doesn't have load values
  df.dropna(axis=0, how='any',inplace=True)
  # filter data by subsystem region
  try:
    df = df[df['nom_subsistema']==regiao].reset_index().drop('index',
                                                             axis=1).copy()
  except:
    pass
  # drops columns about region
  df.drop(labels=['nom_subsistema','id_subsistema'], 
          inplace=True, axis=1,errors='ignore')


  # check if the dataset starts on a friday and go to friday if it does not 
  df = go_to_friday(df)
  # insert missing data from 1st feb'2014
  df, missing_days = id_and_impute(df) 

  # create column with week number 
  df.reset_index(inplace=True,drop=True)
  df['semana'] = (df.index)//7 

  df['Mes'] = df['din_instante'].dt.month
  df['dia semana'] = df['din_instante'].dt.day_name()
  df['dia mes'] = df['din_instante'].dt.day
  df['ano'] = df['din_instante'].dt.year
  
  return df

df = treat_data(df_20XX, regiao='SUDESTE')
df.head(3)

,din_instante,val_cargaenergiamwmed,semana,Mes,dia semana,dia mes,ano
0,2001-01-05,26860.91,0,1,Friday,5,2001
1,2001-01-06,25047.63,0,1,Saturday,6,2001
2,2001-01-07,22943.49,0,1,Sunday,7,2001


# target

In [8]:
def create_target_df(df):
  if df['din_instante'].iloc[0].day_name() != 'Friday':
    # get next friday - begins the operative week
    next_fri = get_friday(df['din_instante'].iloc[0])
    # df starts with the begin of operative week
    df = df[df['din_instante'] >= next_fri].copy()
  # average daily load by operative week
  df_target = pd.DataFrame(data=df.groupby(by=['semana'])[load_col].mean())
  # start day of each operative week
  df_target['Data'] = df.groupby(by=['semana'])[time_col].min()
  return df_target

df_target = create_target_df(df)

# check missing data

In [150]:
df.groupby(['ano'])[time_col].count()

ano
2001    361
2002    365
2003    365
2004    366
2005    365
2006    365
2007    365
2008    366
2009    365
2010    365
2011    365
2012    366
2013    365
2014    365
2015    365
2016    352
2017    365
2018    365
2019    365
2020    366
2021    365
Name: din_instante, dtype: int64

There's one missing day in 2014 and 9 in 2016 (leap year).

In [80]:
# range of every day from 2001 to 2021
time_delta = pd.date_range(start = df.din_instante.iloc[0], end= df.din_instante.iloc[-1],freq='D')
# turn into df
df_time = pd.DataFrame(data={'data':time_delta})
# left join range of data with datas from dataset, missing days will become NaN
df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
# missing days indexes
df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()]
missing_days = df_missing.loc[df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()].index].data
missing_days

4775   2014-02-01
5569   2016-04-05
5570   2016-04-06
5571   2016-04-07
5572   2016-04-08
5573   2016-04-09
5574   2016-04-10
5575   2016-04-11
5576   2016-04-12
5577   2016-04-13
Name: data, dtype: datetime64[ns]

# input missing data

In [ ]:
def imputer(df):

  # range of every day from 2001 to 2021
  time_delta = pd.date_range(start = df.din_instante.iloc[0], end= df.din_instante.iloc[-1],freq='D')
  # turn into df
  df_time = pd.DataFrame(data={'data':time_delta})
  # left join range of data with datas from dataset, missing days will become NaN
  df_missing = df_time.join(df.set_index('din_instante'), on='data', how='left')
  # missing days indexes
  df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()]
  # series of missing days with indexes
  missing_days = df_missing.loc[df_missing.val_cargaenergiamwmed[df_missing.val_cargaenergiamwmed.isnull()].index].data
  
  # missing day to be inputed - feb 1st, 2014
  input_day = df['din_instante'].iloc[missing_days.index[0]]
  # line to be inserter in dataset with load value of day before
  df_day = pd.DataFrame({'din_instante': input_day-pd.Timedelta(1, unit='D'),	
                        'val_cargaenergiamwmed': df[load_col].iloc[missing_days.index[0] -1]},
                        index=[missing_days.index[0] -1])
  # insert missing day
  df_total = pd.concat(objs= [df[:missing_days.index[0]], df_day, df[missing_days.index[0]:]])
  # drop 5 days from apr'2016 so the operative week starts on friday
  #df_total.drop
  #reset index
  return  df_total#.reset_index(drop=True)

imputed_df = imputer(df)


In [75]:
imputed_df.iloc[0]

din_instante             2001-01-05 00:00:00
val_cargaenergiamwmed               26860.91
semana                                   0.0
Mes                                      1.0
dia semana                            Friday
dia mes                                  5.0
ano                                   2001.0
Name: 0, dtype: object